In [ ]:
from molgrid.quadrature import Lebedev


leb = Lebedev() 


In [ ]:

points, weights = leb.get(3)

points, weights

(array([[ 1.,  0.,  0.],
        [-1.,  0.,  0.],
        [ 0.,  1.,  0.],
        [ 0., -1.,  0.],
        [ 0.,  0.,  1.],
        [ 0.,  0., -1.]]),
 array([0.16666667, 0.16666667, 0.16666667, 0.16666667, 0.16666667,
        0.16666667]))

In [ ]:
leb.get_degrees_list()

leb.get_npoints_list()

[6,
 14,
 26,
 38,
 50,
 74,
 86,
 110,
 146,
 170,
 194,
 230,
 266,
 302,
 350,
 434,
 590,
 770,
 974,
 1202,
 1454,
 1730,
 2030,
 2354,
 2702,
 3074,
 3470,
 3890,
 4334,
 4802,
 5294,
 5810]

In [ ]:
from molgrid.quadrature import GaussChebychev

aaaa = GaussChebychev()


x, y = aaaa.finite(9)
#x, y = aaaa.semi_infinite(0.35, 30)

print(y)

print(sum(y))

[0.02999954 0.10853936 0.20561991 0.28415972 0.31415927 0.28415972
 0.20561991 0.10853936 0.02999954]
1.5707963267948968


In [ ]:
from scipy.special import roots_chebyu

xx, w_cheb = roots_chebyu(9)

print(w_cheb)
print(sum(w_cheb)*2)

[0.02999954 0.10853936 0.20561991 0.28415972 0.31415927 0.28415972
 0.20561991 0.10853936 0.02999954]
3.1415926535897936


In [ ]:
import numpy as np
from molgrid.quadrature import GaussChebychev

def wavefunction(r):
    """波函数 psi(r)"""
    return (27 - 18*r + 2*r**2) * np.exp(-r/3)

def integrand(r):
    """被积函数: r^2 * |psi|^2"""
    psi = wavefunction(r)
    return r**2 * psi**2

def calculate_overlap(nshells=50, r_scale=10.0):
    """
    用 Gauss-Chebyshev 求积计算自重叠积分
    
    Parameters
    ----------
    nshells : int
        径向网格点数
    r_scale : float
        径向缩放因子
    
    Returns
    -------
    overlap : float
        自重叠积分值
    normalization : float
        归一化常数
    """
    # 创建 Gauss-Chebyshev 求积对象
    gc = GaussChebychev()
    
    # 获取径向网格点和权重
    # 使用 semi_infinite 方法，r_scale 作为缩放参数
    r, w = gc.semi_infinite(r_scale, nshells)
    
    # 计算积分: 4π * Σ w_i * integrand(r_i)
    overlap = 4 * np.pi * np.sum(w * integrand(r))
    
    # 归一化常数 N = 1/√(overlap)
    normalization = 1.0 / np.sqrt(overlap)
    
    return overlap, normalization

def theoretical_values():
    """理论值 - 需要确认正确的理论值"""
    # 根据题目给的 N = 0.00402142
    N_given = 0.00402142
    overlap_given = 1 / N_given**2
    
    # 或者用符号计算
    from sympy import symbols, exp, integrate, oo, pi
    r_sym = symbols('r', positive=True)
    psi_sym = (27 - 18*r_sym + 2*r_sym**2) * exp(-r_sym/3)
    integrand_sym = 4 * pi * r_sym**2 * psi_sym**2
    overlap_sym = integrate(integrand_sym, (r_sym, 0, oo))
    
    print(f"符号计算结果: ⟨ψ|ψ⟩ = {overlap_sym.evalf():.10f}")
    print(f"对应的 N = {1/np.sqrt(float(overlap_sym.evalf())):.10f}")
    
    return float(overlap_sym.evalf()), 1/np.sqrt(float(overlap_sym.evalf()))

# 测试不同网格点数
print("Gauss-Chebyshev 求积验证")
print("=" * 60)

overlap_theory, norm_theory = theoretical_values()
print(f"理论值:")
print(f"  ⟨ψ|ψ⟩ = {overlap_theory:.10f}")
print(f"  N = {norm_theory:.10f}\n")

print("数值结果:")
print(f"{'nshells':<10} {'⟨ψ|ψ⟩':<20} {'N':<20} {'误差 (%)':<10}")
print("-" * 60)

for n in [10, 20, 30, 40, 50, 60, 80, 100, 150, 200]:
    overlap, norm = calculate_overlap(nshells=n, r_scale=15.0)
    error = abs(overlap - overlap_theory) / overlap_theory * 100
    print(f"{n:<10} {overlap:<20.10f} {norm:<20.10f} {error:<10.2e}")

Gauss-Chebyshev 求积验证
符号计算结果: ⟨ψ|ψ⟩ = 61835.9682006079
对应的 N = 0.0040214199
理论值:
  ⟨ψ|ψ⟩ = 61835.9682006079
  N = 0.0040214199

数值结果:
nshells    ⟨ψ|ψ⟩                N                    误差 (%)    
------------------------------------------------------------
10         61322.2012299903     0.0040382308         8.31e-01  
20         61835.8987766655     0.0040214221         1.12e-04  
30         61835.9637252836     0.0040214200         7.24e-06  
40         61835.9673909899     0.0040214199         1.31e-06  
50         61835.9679853216     0.0040214199         3.48e-07  
60         61835.9681276836     0.0040214199         1.18e-07  
80         61835.9681874148     0.0040214199         2.13e-08  
100        61835.9681971113     0.0040214199         5.65e-09  
150        61835.9682002960     0.0040214199         5.04e-10  
200        61835.9682005519     0.0040214199         9.06e-11  


In [ ]:
import numpy as np
from scipy.special import roots_chebyu

def wavefunction(r):
    """波函数 psi(r)"""
    return (27 - 18*r + 2*r**2) * np.exp(-r/3)

def integrand(r):
    """被积函数: r^2 * |psi|^2"""
    psi = wavefunction(r)
    return r**2 * psi**2

def chebyshev_semi_infinite(n, r_scale=15.0):
    """
    使用 scipy 的 roots_chebyu 生成半无限区间 [0, ∞) 上的 Gauss-Chebyshev 求积
    
    Parameters
    ----------
    n : int
        求积点数
    r_scale : float
        径向缩放因子
    
    Returns
    -------
    r : ndarray
        径向求积点
    w : ndarray
        径向求积权重 (用于 ∫ f(r) dr)
    """
    # 获取第二类 Chebyshev 求积的节点和权重 (区间 [-1, 1])
    # roots_chebyu 返回 (x, w) 其中 w 对应 ∫_{-1}^{1} f(x) √(1-x²) dx
    x, w_cheb = roots_chebyu(n)
    
    # 映射到 [0, ∞): r = r_scale * (1 + x) / (1 - x)
    r = r_scale * (1.0 + x) / (1.0 - x)
    
    # 雅可比行列式: dr/dx
    jacobian = 2.0 * r_scale / (1.0 - x)**2
    
    # 从带权积分转换为普通积分
    # ∫_{-1}^{1} f(x) √(1-x²) dx ≈ Σ w_cheb_i f(x_i)
    # 我们想要 ∫ f(r) dr = ∫ f(r(x)) (dr/dx) dx
    # = ∫ [f(r(x)) (dr/dx) / √(1-x²)] √(1-x²) dx
    # ≈ Σ w_cheb_i * [f(r_i) * (dr/dx)_i / √(1-x_i²)]
    
    # 所以最终权重为: w = w_cheb * jacobian / √(1-x²)
    w = w_cheb * jacobian / np.sqrt(1.0 - x**2)
    
    return r, w

def calculate_overlap_scipy(nshells=50, r_scale=15.0):
    """
    用 scipy 的 roots_chebyu 计算自重叠积分
    """
    r, w = chebyshev_semi_infinite(nshells, r_scale)
    
    # 计算积分: 4π ∫ r² ψ² dr = 4π Σ w_i * [r_i² ψ_i²]
    psi = wavefunction(r)
    overlap = 4 * np.pi * np.sum(w * r**2 * psi**2)
    normalization = 1.0 / np.sqrt(overlap)
    
    return overlap, normalization

def theoretical_values():
    """理论值 - 需要确认正确的理论值"""
    # 根据题目给的 N = 0.00402142
    N_given = 0.00402142
    overlap_given = 1 / N_given**2
    
    # 或者用符号计算
    from sympy import symbols, exp, integrate, oo, pi
    r_sym = symbols('r', positive=True)
    psi_sym = (27 - 18*r_sym + 2*r_sym**2) * exp(-r_sym/3)
    integrand_sym = 4 * pi * r_sym**2 * psi_sym**2
    overlap_sym = integrate(integrand_sym, (r_sym, 0, oo))
    
    print(f"符号计算结果: ⟨ψ|ψ⟩ = {overlap_sym.evalf():.10f}")
    print(f"对应的 N = {1/np.sqrt(float(overlap_sym.evalf())):.10f}")
    
    return float(overlap_sym.evalf()), 1/np.sqrt(float(overlap_sym.evalf()))

# 主程序
print("Gauss-Chebyshev 求积验证 (使用 scipy)")
print("=" * 70)

overlap_theory, norm_theory = theoretical_values()
print(f"\n理论值:")
print(f"  ⟨ψ|ψ⟩ = {overlap_theory:.10f}")
print(f"  N = {norm_theory:.10f}")
print(f"  题目给的 N = 0.00402142")

print("\n数值结果:")
print(f"{'nshells':<10} {'⟨ψ|ψ⟩':<20} {'N':<20} {'误差 (%)':<10} {'r_max':<10}")
print("-" * 70)

for n in [10, 20, 30, 40, 50, 60, 80, 100, 150, 200]:
    overlap, norm = calculate_overlap_scipy(nshells=n, r_scale=15.0)
    r, _ = chebyshev_semi_infinite(n, 15.0)
    error = abs(overlap - overlap_theory) / overlap_theory * 100
    print(f"{n:<10} {overlap:<20.10f} {norm:<20.10f} {error:<10.2e} {np.max(r):<10.2f}")

Gauss-Chebyshev 求积验证 (使用 scipy)
符号计算结果: ⟨ψ|ψ⟩ = 61835.9682006079
对应的 N = 0.0040214199

理论值:
  ⟨ψ|ψ⟩ = 61835.9682006079
  N = 0.0040214199
  题目给的 N = 0.00402142

数值结果:
nshells    ⟨ψ|ψ⟩                N                    误差 (%)     r_max     
----------------------------------------------------------------------
10         61322.2012299903     0.0040382308         8.31e-01   725.61    
20         61835.8987766655     0.0040214221         1.12e-04   2670.96   
30         61835.9637252835     0.0040214200         7.24e-06   5832.18   
40         61835.9673909899     0.0040214199         1.31e-06   10209.26  
50         61835.9679853216     0.0040214199         3.48e-07   15802.18  
60         61835.9681276836     0.0040214199         1.18e-07   22610.97  
80         61835.9681874148     0.0040214199         2.13e-08   39876.10  
100        61835.9681971113     0.0040214199         5.65e-09   62004.64  
150        61835.9682002960     0.0040214199         5.04e-10   138603.46 
200        6